# Global preparations

In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.transforms.functional import to_pil_image
from tqdm.notebook import tqdm

torch.set_float32_matmul_precision('high')
import warnings
import torch._dynamo
torch._dynamo.config.cache_size_limit = 128
warnings.filterwarnings(
    "ignore",
    message=".*has_cuda.*deprecated.*"
)
warnings.filterwarnings(
    "ignore",
    message=".*has_cudnn.*deprecated.*"
)
warnings.filterwarnings(
    "ignore",
    message=".*has_mps.*deprecated.*"
)
warnings.filterwarnings(
    "ignore",
    message=".*has_mkldnn.*deprecated.*"
)

device = 'cuda:7'
summation_dtype = torch.float32
random_seed = 1
torch.manual_seed(random_seed)
np.random.seed(random_seed)
torch.backends.cudnn.enabled = True
print(torch.__version__)

2.2.1


# Text data preparation

Let's read small fineweb fragment

In [7]:
import gdown
url = 'https://drive.google.com/file/d/1vWjyIpU6wvCPtx2OdV_4M3MXX6FsOpxV/view'
output = 'fineweb_texts.txt'
if not os.path.exists(output):
    gdown.download(url, output, quiet=False, fuzzy=True)
size = os.path.getsize(output)
if size < 233 * 1024 * 1024:
    raise RuntimeError(f'Download failed: file size {size/1024/1024:.1f} MB')

In [8]:
CONTEXT_SIZE = 32
# Match spike_QK: BOS at index 256, vocab 257 (bytes 0-255 + BOS)
RAW_VOCAB_SIZE = 256
BOS_ID = RAW_VOCAB_SIZE  # 256
VOCAB_SIZE = RAW_VOCAB_SIZE + 1  # 257

In [9]:
TESTING_LENGTH = 10_000

from spiky.util.text_snippet_sampler import TextSnippetSampler

snippet_sampler = TextSnippetSampler(
    'fineweb_texts.txt',
    CONTEXT_SIZE,
    TESTING_LENGTH,
    device,
    random_seed=random_seed,
)

In [10]:
snippet_sampler.sample_training_batch(2)

tensor([[117, 114, 110,  97, 108, 115,  32, 111, 102,  32,  83, 101, 112, 115,
         105, 115,  10,  65, 108, 108,  32,  80, 117,  98, 108, 105, 115, 104,
         101, 100,  32, 119],
        [116,  32, 116, 104, 101, 115, 101,  32, 112, 111, 115, 116, 115,  32,
          97, 114, 101,  32,  97,  32, 114, 101, 115, 112, 111, 110, 115, 101,
          32, 116, 111,  32]], dtype=torch.int32)

In [11]:
batch = snippet_sampler.sample_training_batch(4)
batch.shape
snippet_sampler.batch_to_text(batch)

['singer and composer, an innovati',
 'he action followed President Oba',
 'in the EU have a European Union ',
 ' was formed in 1979, NSSHA is de']

In [12]:
for test_batch in snippet_sampler.testing_batches_iterator(4):
    print(test_batch.shape)
    print(snippet_sampler.batch_to_text(test_batch))
    break

torch.Size([4, 32])
['cymakers with concrete advice on', 'lose, you can see very clear wha', 'ater with their wings, began to ', 'nd kicks, depending on the situa']


# LUTL1Transformer (MultiHeadLut Q/K + L1Attention + MultiHeadLut V)

In [15]:
# LUTL1Transformer: MultiHeadLut Q, MultiHeadLut K -> L1Attention -> scores * MultiHeadLut V
import torch.nn as nn
from dataclasses import dataclass
from spiky.lutorch.multi_head_lut import MultiHeadLut
from spiky.lutorch.lut_helpers import UncertaintyMode


class L1Attention(nn.Module):
    """Cross-attention module that uses L1 norm of input differences as attention scores.

    Unlike LUTAttention, this module expects **multihead inputs**: each head has its
    own query/key vectors. For each head h and pair (i, j), score[i,j,h] = -||input1[i,h] - input2[j,h]||_1,
    then softmax over keys. No LUT, no pair-processing config, no positional buckets.

    Args:
        causal: If True, apply causal mask so that position k can only attend to positions j <= k.
        attention_temperature: Scale raw scores by 1/temperature before softmax (default: 1.0).
    """

    def __init__(
        self,
        causal: bool = True,
        attention_temperature: float = 1.0,
    ):
        super().__init__()
        self.causal = causal
        self.attention_temperature = float(attention_temperature)

        # Caches for causal path keyed by (batch_size, seq_len, device)
        self._cached_pair_meta = None
        self._cached_batched_rows = None
        self._cached_batched_cols = None
        self._cached_key_indices = None

    def forward(
        self,
        input1: torch.Tensor,
        input2: torch.Tensor,
    ) -> torch.Tensor:
        """Forward pass.

        Args:
            input1: Multihead input tensor of shape [B, S, H, D] (query per head).
            input2: Multihead input tensor of shape [B, S, H, D] (key per head).

        Returns:
            Attention scores tensor of shape [B, S, S, H].
        """
        batch_size, seq_len, H, head_dim = input1.shape
        assert input2.shape == input1.shape, (
            f"input1 and input2 must have the same shape, got {input1.shape} and {input2.shape}"
        )
        device = input1.device

        # [B*S, H, D]
        input1_flat = input1.view(batch_size * seq_len, H, head_dim)
        input2_flat = input2.view(batch_size * seq_len, H, head_dim)

        if self.causal:
            meta = (batch_size, seq_len, device)
            if self._cached_pair_meta != meta:
                rows_local, cols_local = torch.tril_indices(
                    seq_len, seq_len, offset=0, device=device
                )
                offsets = torch.arange(batch_size, device=device) * seq_len
                self._cached_batched_rows = (
                    rows_local.unsqueeze(0) + offsets.unsqueeze(1)
                ).reshape(-1)
                self._cached_batched_cols = (
                    cols_local.unsqueeze(0) + offsets.unsqueeze(1)
                ).reshape(-1)
                self._cached_key_indices = (self._cached_batched_cols % seq_len).contiguous()
                self._cached_pair_meta = meta

            batched_rows = self._cached_batched_rows
            batched_cols = self._cached_batched_cols

            q_vecs = input1_flat[batched_rows]   # [P, H, D]
            k_vecs = input2_flat[batched_cols]   # [P, H, D]
            diff = q_vecs - k_vecs               # [P, H, D]
            l1_per_pair_per_head = diff.abs().sum(dim=-1)  # [P, H]
            raw_scores = -l1_per_pair_per_head

            if self.attention_temperature != 1.0:
                raw_scores = raw_scores / self.attention_temperature

            dense_scores = torch.full(
                (batch_size * seq_len, seq_len, H),
                float("-inf"),
                device=device,
            )
            key_indices = self._cached_key_indices
            dense_scores[batched_rows, key_indices, :] = raw_scores

            attention_scores = dense_scores.view(batch_size, seq_len, seq_len, H)
            attention_scores = torch.softmax(attention_scores, dim=2)
        else:
            # [B, S, S, H, D]: input1[i,h] - input2[j,h] for all (i, j, h)
            input1_expanded = input1.unsqueeze(2).expand(batch_size, seq_len, seq_len, H, head_dim)
            input2_expanded = input2.unsqueeze(1).expand(batch_size, seq_len, seq_len, H, head_dim)
            diff = input1_expanded - input2_expanded  # [B, S, S, H, D]
            l1_grid = diff.abs().sum(dim=-1)  # [B, S, S, H]
            raw_scores = -l1_grid

            if self.attention_temperature != 1.0:
                raw_scores = raw_scores / self.attention_temperature

            attention_scores = torch.softmax(raw_scores, dim=2)

        return attention_scores

from spiky.lutorch.lut_helpers import UncertaintyMode


def _make_attn_lut(c, n_outputs, tables_per_head, n_buckets=1):
    """MultiHeadLut for Q, K, or V (n_buckets=1 for Q/K, no positional)."""
    return MultiHeadLut(
        input_dim=c.embedding_dim,
        n_heads=c.num_heads,
        n_outputs=n_outputs,
        n_anchor_pairs=c.n_anchor_pairs_attn,
        tables_per_head=tables_per_head,
        n_buckets=n_buckets,
        smooth_mode=c.smooth_mode,
        device=c.device,
        connected_anchors_mode=c.connected_anchors_mode,
        random_seed=c.random_seed,
        initial_weights_noise=c.initial_weights_noise,
        uncertainty_mode=c.uncertainty_mode,
    )


@dataclass(frozen=True)
class LUTL1TransformerConfig:
    """Configuration for LUTL1Transformer (MultiHeadLut Q/K + L1Attention + MultiHeadLut V)."""
    vocab_size: int = VOCAB_SIZE
    embedding_dim: int = 32
    num_layers: int = 6
    num_heads: int = 4
    n_anchor_pairs_attn: int = 14
    n_anchor_pairs_ffn: int = 14
    tables_per_head_attn: int = 32
    tables_per_head_value: int = 16
    ffn_tables: int = 16
    dropout: float = 0.0
    smooth_mode: bool = True
    device: object = device
    connected_anchors_mode: bool = False
    random_seed: object = 42
    attention_temperature: float = 0.25
    embedding_temperature: float = 0.1
    initial_weights_noise: float = 0.001
    uncertainty_mode: UncertaintyMode = UncertaintyMode.INVERSE_L1
    max_seq_len: int = 32

    def __post_init__(self):
        assert (self.embedding_dim % self.num_heads) == 0


class LUTL1Transformer(nn.Module):
    """Transformer block: MultiHeadLut Q, MultiHeadLut K -> L1Attention -> scores @ MultiHeadLut V."""

    class Block(nn.Module):
        """Single block: Q/K from LUTs, L1 attention scores, weighted sum with LUT V."""

        def __init__(self, c: LUTL1TransformerConfig):
            super().__init__()
            head_dim = c.embedding_dim // c.num_heads
            self.q_lut = _make_attn_lut(c, head_dim, c.tables_per_head_attn, n_buckets=1)
            self.k_lut = _make_attn_lut(c, head_dim, c.tables_per_head_attn, n_buckets=1)
            self.l1_attn = L1Attention(causal=True, attention_temperature=c.attention_temperature)
            self.value_lut = _make_attn_lut(c, head_dim, c.tables_per_head_value, n_buckets=1)
            self.attn_dropout = nn.Dropout(c.dropout)
            self.ffn = MultiHeadLut(
                input_dim=c.embedding_dim,
                n_heads=1,
                n_outputs=c.embedding_dim,
                n_anchor_pairs=c.n_anchor_pairs_ffn,
                tables_per_head=c.ffn_tables,
                smooth_mode=c.smooth_mode,
                device=c.device,
                connected_anchors_mode=c.connected_anchors_mode,
                random_seed=c.random_seed,
                initial_weights_noise=c.initial_weights_noise,
                uncertainty_mode=c.uncertainty_mode,
            )
            self.ffn_dropout = nn.Dropout(c.dropout)

        def forward(self, z):
            B, S, E = z.shape
            z_flat = z.reshape(-1, E)
            H = self.q_lut.n_heads
            D = E // H

            q = self.q_lut(z_flat).reshape(B, S, H, D)   # [B, S, H, D]
            k = self.k_lut(z_flat).reshape(B, S, H, D)   # [B, S, H, D]
            attn_weights = self.l1_attn(q, k)             # [B, S, S, H]

            v = self.value_lut(z_flat).reshape(B, S, H, D)  # [B, S, H, D]
            attn_out = attn_weights.permute(0, 3, 1, 2) @ v.permute(0, 2, 1, 3)  # [B, H, S, D]
            attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, S, E)  # [B, S, E]

            z = z + self.attn_dropout(attn_out)
            ffn_out = self.ffn(z.reshape(-1, E)).reshape(B, S, -1)
            z = z + self.ffn_dropout(ffn_out)
            return z

    def __init__(self, c: LUTL1TransformerConfig = LUTL1TransformerConfig()):
        super().__init__()
        self.config = c
        with torch.no_grad():
            self.token_embedder = nn.Embedding(c.vocab_size, c.embedding_dim, device=c.device)
            self.token_embedder.weight.copy_(torch.randn(self.token_embedder.weight.shape, device=c.device) * 0.1)
        self.pos_embed = nn.Parameter(torch.zeros(1, c.max_seq_len, c.embedding_dim, device=c.device))
        nn.init.normal_(self.pos_embed, std=0.02)
        self.layers = nn.ModuleList([LUTL1Transformer.Block(c) for _ in range(c.num_layers)])

    def forward(self, tokens):
        z = self.token_embedder(tokens)  # [B, S, E]
        z = z + self.pos_embed[:, : z.size(1), :]
        for layer in self.layers:
            z = layer(z)
        z = z / (z.norm(dim=-1, keepdim=True) + 1e-6)
        logits = z @ self.token_embedder.weight.T / self.config.embedding_temperature
        return logits

In [16]:
lut_transformer = None
optimizer = None
sched = None
if device != 'cpu':
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
lut_transformer = LUTL1Transformer()
print(lut_transformer)

LUTTransformer(
  (token_embedder): Embedding(257, 32)
  (layers): ModuleList(
    (0-5): 6 x Block(
      (cross_attn): LUTCrossAttention(
        (multi_head_lut): MultiHeadLut(
          (lookup): AnchorPairsLookup()
          (projection): LProjection()
        )
      )
      (value_lut): MultiHeadLut(
        (lookup): AnchorPairsLookup()
        (projection): LProjection()
      )
      (attn_dropout): Dropout(p=0.0, inplace=False)
      (ffn): MultiHeadLut(
        (lookup): AnchorPairsLookup()
        (projection): LProjection()
      )
      (ffn_dropout): Dropout(p=0.0, inplace=False)
    )
  )
)


In [61]:
total = sum(p.numel() for p in lut_transformer.parameters())
trainable = sum(p.numel() for p in lut_transformer.parameters() if p.requires_grad)

print("total:", total)
print("trainable:", trainable)
print("frozen:", total - trainable)

total: 201334784
trainable: 201334784
frozen: 0


In [62]:
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

SCALE = 5.0
MAX_RATE = 0.01
def lr_func(t):
    return min(MAX_RATE, SCALE / (1 + t)**0.5)

print(f"Crossover point for LR: {(SCALE / MAX_RATE )**2:,}")

lr = 1.0
optimizer = optim.SGD(lut_transformer.layers.parameters(), lr=lr)
# Match spike_QK: embedder uses global LR scale (0.01); 0.001 was 10x too small
optimizer_embedder = optim.Adam(
    list(lut_transformer.token_embedder.parameters()) + [lut_transformer.pos_embed],
    lr=0.01,
)

steps=1000000
# sched = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=steps
# )
# sched = None
sched = LambdaLR(optimizer, lr_lambda=lr_func)
# sched_embedder = LambdaLR(optimizer_embedder, lr_lambda=lr_func)
# LUTTransformerLutorch has no set_external_learning_rate_hook

Crossover point for LR: 250,000.0


In [63]:
lut_train_losses = []
lut_test_losses = []

In [69]:
import torch
import torch.nn.functional as F

test_batch_size = 128

def generate_text_lut(lut_model, prefix, length, device):
    # Model operates on byte IDs; decode final byte stream as UTF-8 for display.
    # Position 0 = BOS; positions 1..CONTEXT_SIZE-1 = context. We right-align context so
    # position -1 always holds the last byte (matches training where last position = last preceding byte).
    ctx = list(prefix.encode("utf-8"))

    for _ in range(length):
        x = torch.zeros([1, CONTEXT_SIZE], dtype=torch.long, device=device)
        x[0, 0] = BOS_ID
        trunc_ctx = ctx[-(CONTEXT_SIZE - 1):]  # at most CONTEXT_SIZE-1 bytes so we never overwrite BOS
        if len(trunc_ctx) > 0:
            # Right-align: last byte of context at position -1 so logits[:, -1, :] is conditioned correctly
            x[0, -len(trunc_ctx):] = torch.tensor(trunc_ctx, dtype=torch.long, device=device)

        logits = lut_model(x)
        # Exclude BOS from sampling: generated stream should be UTF-8 bytes only.
        probs = torch.softmax(logits[:, -1, :RAW_VOCAB_SIZE], dim=-1)[0]
        next_id = torch.multinomial(probs, 1).item()
        ctx.append(next_id)

    ctx_bytes = [c for c in ctx if 0 <= c < 256]
    return bytes(ctx_bytes).decode("utf-8", errors="replace")

def evaluate_model(model, sampler, B, last_position_only=False):
    """last_position_only=True matches spike_QK (loss only at last token; expect ~6.48 untrained)."""
    model.eval()
    losses = []
    device = next(model.parameters()).device

    with torch.no_grad():
        for batch in sampler.testing_batches_iterator(B):   # [B, C]
            # Prepend BOS at position 0 (match spike_QK)
            inp = torch.empty(batch.shape[0], batch.shape[1], dtype=torch.long, device=batch.device)
            inp[:, 0] = BOS_ID
            inp[:, 1:] = batch[:, :-1].long()
            tgt = batch.long()

            logits = model(inp)   # [B, 32, 257]

            B_, T, V = logits.shape
            if last_position_only:
                loss = F.cross_entropy(logits[:, -1, :], tgt[:, -1], reduction='mean')
                losses.append(loss.item())
            else:
                loss = F.cross_entropy(
                    logits.reshape(B_ * T, V),
                    tgt.reshape(B_ * T),
                    reduction='none'
                ).sum()
                losses.append(loss.item() / (CONTEXT_SIZE * B))

    # ---- small generation demo ----
    prefix = "Once upon a time "
    gen = generate_text_lut(model, prefix, length=80, device=device)
    print("\n[GEN]:", gen, "\n")

    model.train()
    return sum(losses) / len(losses) #  / (CONTEXT_SIZE * B)

In [ ]:
# Use last_position_only=True to match spike_QK (expect ~6.48 untrained). False = mean over all positions (~5.4).
test_loss = evaluate_model(lut_transformer, snippet_sampler, test_batch_size, last_position_only=True)
test_loss


[GEN]: Once upon a time ECQÓ+cn9¦÷6aaÁJ5	þrnÂ@I,²ÇÍKem
ÎK½ÙÉuYöÔÀ

S®$F¸0
á-ô
÷@nÉJS1ª 



5.444685253915908

In [ ]:
test_every=1000
train_loss = None
alpha = 0.01
batch_size = 128

pbar = tqdm(total=steps)
lut_transformer.train()

for step in range(0, steps + 1):
    x = snippet_sampler.sample_training_batch(batch_size)   # [B, 32] bytes
    x = x.long() if x.dtype != torch.long else x
    # Match spike_QK: position 0 = BOS, positions 1..31 = x[:,0..30]; targets = all 32 bytes
    inp = torch.empty(batch_size, x.shape[1], dtype=torch.long, device=x.device)
    inp[:, 0] = BOS_ID
    inp[:, 1:] = x[:, :-1]
    tgt = x

    logits = lut_transformer(inp)      # [B, C-1, 256]
    B, T, V = logits.shape

    # Train on full sequence with sum reduction (gradient from all positions)
    loss = F.cross_entropy(
        logits.reshape(B * T, V),
        tgt.reshape(B * T),
        reduction="sum",
    )

    optimizer.zero_grad()
    optimizer_embedder.zero_grad()
    loss.backward()
    optimizer.step()
    optimizer_embedder.step()
    if sched is not None:
        for _ in range(x.shape[0]):
            sched.step()
            # sched_embedder.step()
        # sched.step()
        # sched_embeddder.step()

    # Reported train loss: match spike_QK exactly (last position only, -log(p + 1e-10).mean())
    with torch.no_grad():
        probs_last = F.softmax(logits[:, -1, :], dim=-1)
        loss_value = (
            -torch.log(probs_last.gather(1, tgt[:, -1:]).squeeze(1) + 1e-10).mean().item()
        )
    train_loss = loss_value if train_loss is None else (1 - alpha) * train_loss + alpha * loss_value
    pbar.update(1)
    if step % 10 == 0:
        pbar.set_description(f"loss={train_loss:.4f}, lr {lr if sched is None else sched.get_last_lr()[0]:.8f}")

    if step % test_every == 0:
        test_loss = evaluate_model(lut_transformer, snippet_sampler, test_batch_size, last_position_only=True)
        if len(lut_train_losses) == 0 or step > 0:
            lut_train_losses.append(train_loss)
            lut_test_losses.append(test_loss)
        print(f"[TEST] step {step}: loss={test_loss:.4f}")
#         if step > 0 and batch_size < 384:
#             print(f"batch_size {batch_size} -> {batch_size + 32}")
#             batch_size += 32

  0%|          | 0/1000000 [00:00<?, ?it/s]


[GEN]: Once upon a time |<#Ñµø[fm\d½íY;8á;¢j6Z3xðvªp ­øÕþ¿Òw~ab¡
XÞ¾Ð´ÉËIîµ8²¢^Ìÿ±V| 

[TEST] step 0: loss=5.2086

[GEN]: Once upon a time and exteti 15 votality ats belia, butant warÞy sciet anal outave leprable hanan  

[TEST] step 1000: loss=1.9690

[GEN]: Once upon a time use/folling Hach 1hal and 3 of this abilition, Devit C
Yow Colly, the Parch 12 

[TEST] step 2000: loss=1.8720

[GEN]: Once upon a time so; cha Kizcomes two GSSH/2inible Read Gas-body composing it his ately.
Offity m 

[TEST] step 3000: loss=1.8002


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# assume train_losses and test_losses are Python lists of equal length

steps = [i * 1000 for i in range(len(lut_train_losses))]

plt.figure(figsize=(6,4))
plt.plot(steps, lut_train_losses, label="train")
plt.plot(steps, lut_test_losses, label="test")
plt.ylim(top=2.0)
plt.xlabel("steps")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [65]:
test_loss

5.505621819556514

In [14]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

profile_steps = 200

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    for step in range(profile_steps):

        with record_function("sample_batch"):
            x = snippet_sampler.sample_training_batch(batch_size)
            x = x.long() if x.dtype != torch.long else x
            inp = torch.empty(batch_size, x.shape[1], dtype=torch.long, device=x.device)
            inp[:, 0] = BOS_ID
            inp[:, 1:] = x[:, :-1]
            tgt = x

        with record_function("forward"):
            logits = lut_transformer(inp)

        with record_function("loss"):
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, V),
                tgt.reshape(B * T),
                reduction="none"
            ).sum()

        with record_function("backward+step"):
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        with record_function("scheduler"):
            if sched is not None:
                for _ in range(x.shape[0]):
                    sched.step()

        prof.step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=40))
prof.export_chrome_trace("trace.json")

STAGE:2026-03-05 01:41:34 304266:304266 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-03-05 01:42:11 304266:304266 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-03-05 01:42:12 304266:304266 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         1.52%     757.733ms        25.73%       12.834s      64.170ms       0.000us         0.00%       20.728s     103.642ms      18.75 Kb      18.75 Kb     713.63 Gb    -741.48 G

In [26]:
steps=1000000


In [17]:
from spiky.lutorch.lut_helpers import logarithmic_pe_buckets, rpe_matrix

In [29]:
pe_buckets = logarithmic_pe_buckets(8, 32, device)

In [30]:
pe_buckets

tensor([0, 1, 2, 3, 4, 4, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7], device='cuda:7')

In [32]:
rpe_matrix(pe_buckets, 32, device).T

tensor([[0, 1, 2,  ..., 7, 7, 7],
        [0, 0, 1,  ..., 7, 7, 7],
        [0, 0, 0,  ..., 7, 7, 7],
        ...,
        [0, 0, 0,  ..., 0, 1, 2],
        [0, 0, 0,  ..., 0, 0, 1],
        [0, 0, 0,  ..., 0, 0, 0]], device='cuda:7')

In [36]:
causal_mask = torch.tril(torch.ones(32, 32, device=device), diagonal=-1).unsqueeze(0).unsqueeze(-1)
causal_mask == 0

tensor([[[[ True],
          [ True],
          [ True],
          ...,
          [ True],
          [ True],
          [ True]],

         [[False],
          [ True],
          [ True],
          ...,
          [ True],
          [ True],
          [ True]],

         [[False],
          [False],
          [ True],
          ...,
          [ True],
          [ True],
          [ True]],

         ...,

         [[False],
          [False],
          [False],
          ...,
          [ True],
          [ True],
          [ True]],

         [[False],
          [False],
          [False],
          ...,
          [False],
          [ True],
          [ True]],

         [[False],
          [False],
          [False],
          ...,
          [False],
          [False],
          [ True]]]], device='cuda:7')

In [37]:
attention_scores = torch.zeros([2, 32, 32, 1], device=device)

In [38]:
attention_scores = attention_scores.masked_fill(causal_mask == 0, float('-inf')).squeeze(3)    

In [40]:
F.softmax(attention_scores[:, 0])

/tmp/ipykernel_367987/2957606799.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(attention_scores[:, 0])


tensor([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan]], device='cuda:7')

In [25]:
attention_scores.shape

torch.Size([2, 32, 32, 1])

In [18]:
seq_len = 32
batch_size = 3
rows_local, cols_local = torch.tril_indices(
    seq_len, seq_len, offset=-1, device=device
)  # [num_pairs_single]

offsets = torch.arange(batch_size, device=device) * seq_len  # [B]
_cached_batched_rows = (
    rows_local.unsqueeze(0) + offsets.unsqueeze(1)
).reshape(-1)  # [P], where P = B * num_pairs_single
_cached_batched_cols = (
    cols_local.unsqueeze(0).expand(batch_size, -1)
).reshape(-1)  # [P]

# Within-sequence key indices for scattering into [B*S, S, H]
_cached_key_indices = (_cached_batched_cols % seq_len).contiguous()  # [P]

pe_buckets = logarithmic_pe_buckets(8, seq_len, device)
rpe = rpe_matrix(pe_buckets, seq_len, device)  # [S, S]
rpe_pairs = rpe[rows_local, cols_local]  # [num_pairs_single]
_cached_bucket_indices = rpe_pairs.repeat(batch_size).contiguous()  # [P]

In [24]:
_cached_batched_rows[:16], _cached_batched_cols[:16]

(tensor([1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 5, 6], device='cuda:7'),
 tensor([0, 0, 1, 0, 1, 2, 0, 1, 2, 3, 0, 1, 2, 3, 4, 0], device='cuda:7'))

In [26]:
_cached_batched_rows[-16:], _cached_batched_cols[-16:]

(tensor([95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95],
        device='cuda:7'),
 tensor([15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
        device='cuda:7'))

In [16]:
cols_local

tensor([ 0,  0,  1,  0,  1,  2,  0,  1,  2,  3,  0,  1,  2,  3,  4,  0,  1,  2,
         3,  4,  5,  0,  1,  2,  3,  4,  5,  6,  0,  1,  2,  3,  4,  5,  6,  7,
         0,  1,  2,  3,  4,  5,  6,  7,  8,  0,  1,  2,  3,  4,  5,  6,  7,  8,
         9,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,
        12,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,  0,  1,  2,
         3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,
         8,  9, 10, 11, 12, 13, 14, 15, 16,  0,  1,  2,  3,  4,  5,  6,  7,  8,
         9, 10, 11, 12, 13, 14, 15, 16, 17,  0,  1,  2,  3,  4,  5,  6,  7,  8,
         9, 10, 11, 12, 13, 14, 15, 16, 17, 18,  0,  1,  2,  3,  4,  5,  6,  7,
         8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11, 12, 13, 